# __MODEL_LABEL_MARKDOWN__: optional editor

Select any published candidate version for this SQL model, or leave the
package unset to open the latest. Publishing an editor session creates an
immutable `EDITOR_EDIT` child package.


In [ ]:
DATABASE_MODE = __DATABASE_MODE_LITERAL__  # "local" or "remote"
RUNTIME_MODULE = __RUNTIME_MODULE_LITERAL__  # e.g. "work_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = __EXPECTED_REMOTE_DATABASE_LITERAL__
ALLOW_REMOTE_WRITES = False

MODEL_NAME = "__MODEL_NAME__"  # Set to None to select by label only.
MODEL_LABEL = "__MODEL_LABEL__"
DEPLOYMENT_SLOT = "__DEPLOYMENT_SLOT__"
PACKAGE_VERSION = None  # None selects the latest listed package.
EDIT_REASON = ""


In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "pricing_models").is_dir()
)

from superglm.editor import EditorSession

from pricing_pipeline.notebook import (
    connect,
    list_candidate_versions,
    load_registered_model,
    open_candidate,
    publish_edits,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/__PACKAGE_NAME__"
FRAME_ARTIFACT_PATH = MODEL_DIR / ".local" / "model_frame.joblib"
GROUPING_ARTIFACT_PATH = MODEL_DIR / ".local" / "routine_groupings.joblib"


## Connect, resolve the model label, and list versions


In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)


In [ ]:
model = load_registered_model(
    pricing,
    model_name=MODEL_NAME,
    model_label=MODEL_LABEL,
    deployment_slot=DEPLOYMENT_SLOT,
    source_root=MODEL_DIR,
)
versions = list_candidate_versions(pricing, model=model)
display(versions)


## Select and open the exact candidate


In [ ]:
if versions.empty:
    raise LookupError("No candidate package versions were found.")
selected_package_version = (
    int(versions.iloc[0]["Package"])
    if PACKAGE_VERSION is None
    else int(PACKAGE_VERSION)
)
if selected_package_version not in set(versions["Package"].astype(int)):
    raise ValueError("PACKAGE_VERSION is not in the displayed candidate list.")
reviewed = open_candidate(
    pricing,
    model=model,
    package_version=selected_package_version,
)
display(reviewed.technical)


## Open the live editor


In [ ]:
editor_session = EditorSession.from_model(
    reviewed.bundle.fitted_model,
    train_data=(
        reviewed.bundle.X,
        reviewed.bundle.y,
        reviewed.bundle.sample_weight,
        reviewed.bundle.offset,
    ),
    cv_report=reviewed.bundle.cv_report,
)
display(editor_session.widget())


## Preview the edited model without publishing


In [ ]:
edited_model = editor_session.to_model()
edited_model


## Publish the retained editor session


In [ ]:
if not EDIT_REASON.strip():
    raise ValueError("Describe the market or underwriting edit.")
edited = publish_edits(
    pricing,
    candidate=reviewed,
    editor_session=editor_session,
    reason=EDIT_REASON,
)
display({
    "Kind": edited.model_kind,
    "Package": edited.package_version,
    "Parent package ID": edited.parent_rate_package_id,
    "State": edited.package_status,
    "Reused equivalent": edited.deduplicated,
})
